# Chuong 6: Tong Ket & So Sanh
**Nhom 11**

In [ ]:
import sys, os
os.environ['TF_ENABLE_ONEDNN_OPTS'] = '0'
sys.path.insert(0, os.path.abspath('..'))
import warnings; warnings.filterwarnings('ignore')
import pickle
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from sklearn.metrics import roc_auc_score, roc_curve
from src.config import *
from src.utils import (setup_plot_style, savefig, metrics_table,
                        compute_all_metrics, plot_radar_chart, plot_training_time)
setup_plot_style()
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
with open(PROCESSED_DIR/'bt1_predictions.pkl','rb') as f: bt1=pickle.load(f)
with open(PROCESSED_DIR/'training_times.pkl','rb') as f: times=pickle.load(f)
with open(PROCESSED_DIR/'bt2_vae_results.pkl','rb') as f: bt2=pickle.load(f)
y = bt1['y']
print('All results loaded!')
for n,k in [('LightGBM','lgbm'),('DAE','dae'),('ResNet','resnet'),('W&D','wnd'),('DL Ensemble','ensemble')]:
    print(f'  {n:12s}: AUC={roc_auc_score(y,bt1[k]):.4f}')
print(f'  Beta-VAE     : AUC={bt2["auc"]:.4f}')

## Bang Tong Hop Ket Qua

In [ ]:
pd_d = {'LightGBM':bt1['lgbm'],'DAE':bt1['dae'],'ResNet':bt1['resnet'],'W&D':bt1['wnd'],'DL Ensemble':bt1['ensemble']}
df_m = metrics_table(pd_d, y)
print('=== BT1: Credit Risk Assessment ===')
print(df_m[['AUC','Precision','Recall','F1-Score','Avg Precision']].to_string())
print()
print('=== BT2: Anomaly Detection (Beta-VAE) ===')
print(f'  AUC          : {bt2["auc"]:.4f}')
print(f'  Avg Precision: {bt2["ap"]:.4f}')
print(f'  Precision@p95: {bt2["precision"]:.4f}')
print(f'  Recall@p95   : {bt2["recall"]:.4f}')
print(f'  F1@p95       : {bt2["f1"]:.4f}')

## Hinh 7.2 – Radar Chart So Sanh Models

In [ ]:
mc = {
    'DL Ensemble': compute_all_metrics(y, bt1['ensemble']),
    'LightGBM':    compute_all_metrics(y, bt1['lgbm']),
    'ResNet':      compute_all_metrics(y, bt1['resnet']),
}
plot_radar_chart(mc, title='Hinh 7.2: So Sanh Model – Radar Chart', save_name='fig_7_2_radar_chart.png')

## Hinh 7.3 – Training Time

In [ ]:
times['Beta-VAE'] = bt2.get('time', 120)
plot_training_time(times, title='Hinh 7.3: Thoi Gian Training', save_name='fig_7_3_training_time.png')

## Tong Hop 2 Bai Toan

In [ ]:
fig = plt.figure(figsize=(20, 8))
fig.patch.set_facecolor('#1a1a2e')
gs = gridspec.GridSpec(1, 3, figure=fig, wspace=0.32)

all_aucs = {n:roc_auc_score(y,bt1[k]) for n,k in [('LGBM','lgbm'),('DAE','dae'),('ResNet','resnet'),('W&D','wnd'),('Ensemble','ensemble')]}
all_aucs['Beta-VAE'] = bt2['auc']
c6 = [PALETTE['lgbm'],PALETTE['dae'],PALETTE['resnet'],PALETTE['wnd'],PALETTE['ensemble'],PALETTE['vae']]

ax1 = fig.add_subplot(gs[0,0])
bars = ax1.bar(all_aucs.keys(), all_aucs.values(), color=c6, edgecolor='white', width=0.55, zorder=3)
for bar,val in zip(bars,all_aucs.values()):
    ax1.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.003,
             f'{val:.4f}', ha='center', va='bottom', fontsize=9, color='white', fontweight='bold')
ax1.set_ylim(0.45, max(all_aucs.values())+0.05)
ax1.set_title('AUC – BT1 & BT2', color='white', fontsize=11)
ax1.set_facecolor('#16213e'); ax1.tick_params(axis='x', rotation=25)

ax2 = fig.add_subplot(gs[0,2])
mn = ['AUC','Precision','Recall','F1-Score']
ev = [df_m.loc['DL Ensemble',m] for m in mn]
lv = [df_m.loc['LightGBM',m] for m in mn]
x_pos = np.arange(len(mn)); w=0.35
ax2.bar(x_pos-w/2, ev, w, label='DL Ensemble', color=PALETTE['ensemble'], edgecolor='white')
ax2.bar(x_pos+w/2, lv, w, label='LightGBM',    color=PALETTE['lgbm'],     edgecolor='white')
ax2.set_xticks(x_pos); ax2.set_xticklabels(mn, fontsize=9)
ax2.set_title('DL Ensemble vs LightGBM', color='white', fontsize=11)
ax2.set_ylim(0, 1.1); ax2.legend(fontsize=9); ax2.set_facecolor('#16213e')

fig.suptitle('Tong Ket – Nhom 11', fontsize=15, fontweight='bold', color='white')
plt.tight_layout()
savefig('fig_tong_ket.png')
plt.show()

## Ket Luan & Huong Phat Trien

**BT1:** DL Ensemble canh tranh voi LightGBM, Focal Loss hieu qua voi imbalance 8.1%

**BT2:** Beta-VAE voi beta=1.5 tranh KL collapse, phat hien bat thuong qua reconstruction error

**Huong phat trien:** FT-Transformer | Graph Neural Network | Multi-task Learning | SHAP

In [ ]:
print('=== DU AN HOAN THANH ===')
figs = list(FIGURES_DIR.glob('*.png'))
print(f'Figures: {len(figs)}')
for f in sorted(figs): print(f'  {f.name}')
print()
print('Final Summary:')
print(f'  LightGBM    : {roc_auc_score(y,bt1["lgbm"]):.4f}')
print(f'  DL Ensemble : {roc_auc_score(y,bt1["ensemble"]):.4f}')
print(f'  Beta-VAE    : {bt2["auc"]:.4f}')